# HW1 – 2025 – Simulation | Phase 2: Full Criteria
## Domain: Video Games (PC & PlayStation)

**Author:** Georgios Kitsakis  
**Institution:** Athens University of Economics and Business (AUEB)

---

**Phase 2** adds harder, richer criteria on top of the simple platform/price/age filters.
The goal is to give LDA more distinctive signals per segment and see if segment discovery improves.

| # | Segment | Full like condition |
|---|---|---|
| 1 | **PC Purist** | PC/BOTH + Metacritic ≥ 75 + playtime ≥ 20h (bonus: Strategy/RPG + Meta 70–74 → 70%) |
| 2 | **PlayStation Fan** | PS/BOTH + Metacritic ≥ 70 (bonus: PS exclusive Action/Adventure → 65%) |
| 3 | **Cross-Platform Gamer** | BOTH + Metacritic ≥ 72 + multiplayer (bonus: Meta 68–71 + multi → 60%) |
| 4 | **Budget Gamer** | price ≤ €20 + Metacritic ≥ 55 (bonus: price ≤ €10 + Meta ≥ 50 → 75%) |
| 5 | **Casual / Family Gamer** | PEGI '3'/'7' + playtime ≤ 15h + Metacritic ≥ 60 (bonus: Puzzle/Racing → 70%) |

→ See **Phase 1** notebook for simple criteria baseline.

In [1]:
!pip install tomotopy -q

In [2]:
import random
import csv
import numpy as np
import pandas as pd
import tomotopy as tp
from dataclasses import dataclass
from typing import List, Tuple
import warnings
warnings.filterwarnings('ignore')

SEG_NAMES = {
    1: 'PC Purist',
    2: 'PlayStation Fan',
    3: 'Cross-Platform Gamer',
    4: 'Budget Gamer',
    5: 'Casual / Family Gamer'
}

---
## 1. `generate_entities()`

Each game has **10 distinct attributes**: title, platform, genre, price_eur, metacritic,
avg_playtime_h, is_multiplayer, is_exclusive, age_rating, release_year.

In [3]:
@dataclass
class VideoGame:
    title: str
    platform: str
    genre: str
    price_eur: float
    metacritic: int
    avg_playtime_h: float
    is_multiplayer: bool
    is_exclusive: bool
    age_rating: str
    release_year: int

In [4]:
def generate_entities(
    game_num: int = 200,
    genre_options: List[str] = ['Action', 'RPG', 'Sports', 'Strategy', 'Horror',
                                 'Adventure', 'Simulation', 'Fighting', 'Puzzle', 'Racing'],
    platform_options: List[str] = ['PC', 'PS', 'BOTH'],
    platform_distro: List[float] = [0.35, 0.30, 0.35],
    price_gaussian_params: Tuple[float, float] = (35, 18),
    metacritic_gaussian_params: Tuple[float, float] = (68, 15),
    playtime_gaussian_params: Tuple[float, float] = (25, 20),
    multiplayer_prob: float = 0.45,
    age_ratings: List[str] = ['3', '7', '12', '16', '18'],
    year_range: Tuple[int, int] = (2010, 2024)
) -> List[VideoGame]:
    """
    Generates a list of synthetic video game entities.

    Each game has 10 distinct attributes:
        title           – unique string identifier (e.g. 'Game_0001')
        platform        – 'PC', 'PS', or 'BOTH' (sampled from platform_distro)
        genre           – primary genre from genre_options
        price_eur       – retail price in EUR (Gaussian, clipped to [5, 80])
        metacritic      – Metacritic score 0–100 (Gaussian, clipped)
        avg_playtime_h  – average playtime in hours (Gaussian, clipped to [1, 200])
        is_multiplayer  – True if the game supports online multiplayer
        is_exclusive    – True if platform-exclusive (platform != 'BOTH')
        age_rating      – PEGI rating: '3', '7', '12', '16', or '18'
        release_year    – release year, sampled uniformly in year_range

    Args:
        game_num (int):                    Number of games to generate. Default: 200.
        genre_options (List[str]):         Available genres.
        platform_options (List[str]):      Platform choices.
        platform_distro (List[float]):     Probability distribution over platforms.
        price_gaussian_params (Tuple):     (mean, std) for price in EUR.
        metacritic_gaussian_params (Tuple):(mean, std) for Metacritic score.
        playtime_gaussian_params (Tuple):  (mean, std) for avg playtime in hours.
        multiplayer_prob (float):          Probability of multiplayer support.
        age_ratings (List[str]):           PEGI rating options.
        year_range (Tuple[int, int]):      (min_year, max_year) for release year.

    Returns:
        List[VideoGame]: A list of generated VideoGame objects.
    """
    games = []
    for i in range(game_num):
        platform_idx = np.random.choice(len(platform_options), p=platform_distro)
        platform     = platform_options[platform_idx]
        genre        = random.choice(genre_options)
        price        = round(float(np.clip(random.gauss(*price_gaussian_params), 5.0, 80.0)), 2)
        metacritic   = int(np.clip(random.gauss(*metacritic_gaussian_params), 0, 100))
        playtime     = round(float(np.clip(random.gauss(*playtime_gaussian_params), 1.0, 200.0)), 1)
        is_multi     = random.random() < multiplayer_prob
        is_excl      = platform != 'BOTH'
        age_rating   = random.choice(age_ratings)
        year         = random.randint(*year_range)
        games.append(VideoGame(
            title=f'Game_{i+1:04d}', platform=platform, genre=genre,
            price_eur=price, metacritic=metacritic, avg_playtime_h=playtime,
            is_multiplayer=is_multi, is_exclusive=is_excl,
            age_rating=age_rating, release_year=year
        ))
    return games

In [5]:
games = generate_entities(game_num=200)
print(f'Generated {len(games)} games.')
print(vars(games[0]))
print('Platform distribution:', pd.Series([g.platform for g in games]).value_counts().to_dict())

Generated 200 games.
{'title': 'Game_0001', 'platform': 'BOTH', 'genre': 'Adventure', 'price_eur': 35.01, 'metacritic': 77, 'avg_playtime_h': 5.7, 'is_multiplayer': True, 'is_exclusive': False, 'age_rating': '3', 'release_year': 2022}
Platform distribution: {'BOTH': 73, 'PC': 68, 'PS': 59}


---
## 2. `generate_users()`

Same 5 segments as Phase 1 — one helper per segment, all called from `generate_users()`.

In [6]:
@dataclass
class User:
    segment: int
    age: int
    gender: str

In [7]:
def generate_users_segment1(user_num: int = 200) -> List[User]:
    """
    Generates users for Segment 1: PC Purist.

    Characteristics:
    - Dedicated PC gamers who only care about games on PC or BOTH platforms
    - Demand high Metacritic scores (>=75) and long playtime (>=20h)
    - Favourite genres: Strategy and RPG
    - 50% Male, 50% Female
    - Age: Gaussian(30, 6)

    Args:
        user_num (int): Number of users. Default: 200.
    Returns:
        List[User]: Segment 1 users.
    """
    return [User(segment=1, age=max(10, int(random.gauss(30, 6))),
                 gender=random.choice(['M','F'])) for _ in range(user_num)]


def generate_users_segment2(user_num: int = 200) -> List[User]:
    """
    Generates users for Segment 2: PlayStation Fan.

    Characteristics:
    - Console loyalists who only buy games for their PlayStation (PS or BOTH)
    - Strongly prefer PlayStation exclusives in Action/Adventure genres
    - Demand Metacritic >= 70
    - 50% Male, 50% Female
    - Age: Gaussian(25, 7)

    Args:
        user_num (int): Number of users. Default: 200.
    Returns:
        List[User]: Segment 2 users.
    """
    return [User(segment=2, age=max(10, int(random.gauss(25, 7))),
                 gender=random.choice(['M','F'])) for _ in range(user_num)]


def generate_users_segment3(user_num: int = 200) -> List[User]:
    """
    Generates users for Segment 3: Cross-Platform Gamer.

    Characteristics:
    - Own both PC and PlayStation; only buy games on BOTH platforms
    - Require strong multiplayer support and Metacritic >= 72
    - 50% Male, 50% Female
    - Age: Gaussian(22, 5)

    Args:
        user_num (int): Number of users. Default: 200.
    Returns:
        List[User]: Segment 3 users.
    """
    return [User(segment=3, age=max(10, int(random.gauss(22, 5))),
                 gender=random.choice(['M','F'])) for _ in range(user_num)]


def generate_users_segment4(user_num: int = 200) -> List[User]:
    """
    Generates users for Segment 4: Budget Gamer.

    Characteristics:
    - Price-first: any platform/genre as long as price <= 20 EUR
    - Minimum quality bar: Metacritic >= 55
    - 50% Male, 50% Female
    - Age: Gaussian(20, 8)

    Args:
        user_num (int): Number of users. Default: 200.
    Returns:
        List[User]: Segment 4 users.
    """
    return [User(segment=4, age=max(10, int(random.gauss(20, 8))),
                 gender=random.choice(['M','F'])) for _ in range(user_num)]


def generate_users_segment5(user_num: int = 200) -> List[User]:
    """
    Generates users for Segment 5: Casual / Family Gamer.

    Characteristics:
    - Occasional players wanting short, easy, family-friendly games
    - Require PEGI '3' or '7', playtime <= 15h, Metacritic >= 60
    - Favourite genres: Puzzle and Racing
    - Any platform works
    - 50% Male, 50% Female
    - Age: Gaussian(38, 10)

    Args:
        user_num (int): Number of users. Default: 200.
    Returns:
        List[User]: Segment 5 users.
    """
    return [User(segment=5, age=max(10, int(random.gauss(38, 10))),
                 gender=random.choice(['M','F'])) for _ in range(user_num)]

In [8]:
def generate_users(user_num: int = 1000) -> List[User]:
    """
    Generates a population of users spread across 5 distinct customer segments.
    Users are distributed equally (user_num // 5 per segment).

    Internally calls:
        generate_users_segment1() – PC Purist
        generate_users_segment2() – PlayStation Fan
        generate_users_segment3() – Cross-Platform Gamer
        generate_users_segment4() – Budget Gamer
        generate_users_segment5() – Casual / Family Gamer

    Args:
        user_num (int): Total number of users to generate. Default: 1000.

    Returns:
        List[User]: A shuffled list of User objects tagged with segment id (1–5).
    """
    n = user_num // 5
    users = (
        generate_users_segment1(n) +
        generate_users_segment2(n) +
        generate_users_segment3(n) +
        generate_users_segment4(n) +
        generate_users_segment5(n)
    )
    random.shuffle(users)
    return users

In [9]:
users = generate_users(user_num=1000)
counts = pd.Series([u.segment for u in users]).value_counts().sort_index()
for seg, cnt in counts.items():
    print(f'  Segment {seg} – {SEG_NAMES[seg]:25s}: {cnt} users')

  Segment 1 – PC Purist                : 200 users
  Segment 2 – PlayStation Fan          : 200 users
  Segment 3 – Cross-Platform Gamer     : 200 users
  Segment 4 – Budget Gamer             : 200 users
  Segment 5 – Casual / Family Gamer    : 200 users


---
## 3. `generate_ratings()` — Full Criteria

Each segment has its own helper with **richer, harder conditions**:
platform/price/age check is still required, but Metacritic thresholds,
playtime limits, and genre bonuses are now enforced.
After the ground-truth is decided, the rating is flipped with probability `noise`.

In [10]:
def _make_row(user, game, rating, reason):
    return {'segment': user.segment, 'age': user.age, 'gender': user.gender,
            'game': game.title, 'platform': game.platform, 'genre': game.genre,
            'price_eur': game.price_eur, 'metacritic': game.metacritic,
            'avg_playtime_h': game.avg_playtime_h, 'is_multiplayer': game.is_multiplayer,
            'is_exclusive': game.is_exclusive, 'age_rating': game.age_rating,
            'release_year': game.release_year, 'rating': rating, 'reason': reason}


def generate_ratings_segment1(users, games, pairs, noise):
    """
    Generates ratings for Segment 1: PC Purist — Full Criteria.

    Criteria:
    - AVG_RATING >= 75 (Metacritic)
    - AVG_PLAYTIME >= 20h
    - Game must be on PC or BOTH platform
    - BONUS: Strategy/RPG + Metacritic 70-74 → 70% like chance
    - noise% chance of rating flip

    Args:
        users: full user list
        games: full game list
        pairs: (user_idx, game_idx) tuples for this segment
        noise: flip probability
    Returns:
        list of row dicts
    """
    rows = []
    for u_idx, g_idx in pairs:
        user, game = users[u_idx], games[g_idx]
        rating, reason = -1, ''
        if game.platform not in ('PC', 'BOTH'):
            reason = 'Not on PC'
        elif game.metacritic >= 75 and game.avg_playtime_h >= 20:
            rating = 1
            reason = 'PC + high Metacritic + long playtime'
        elif game.genre in ('Strategy', 'RPG') and 70 <= game.metacritic < 75:
            if random.random() < 0.70:
                rating = 1; reason = 'Favourite genre + decent Metacritic (bonus)'
            else:
                reason = 'Favourite genre but Metacritic not convincing'
        else:
            reason = 'PC but Metacritic or playtime too low'
        if random.random() < noise:
            rating *= -1; reason += ' [NOISE FLIP]'
        rows.append(_make_row(user, game, rating, reason))
    return rows


def generate_ratings_segment2(users, games, pairs, noise):
    """
    Generates ratings for Segment 2: PlayStation Fan — Full Criteria.

    Criteria:
    - AVG_RATING >= 70 (Metacritic)
    - Game must be on PS or BOTH platform
    - BONUS: PS exclusive Action/Adventure → 65% like chance
    - noise% chance of rating flip

    Args:
        users: full user list
        games: full game list
        pairs: (user_idx, game_idx) tuples for this segment
        noise: flip probability
    Returns:
        list of row dicts
    """
    rows = []
    for u_idx, g_idx in pairs:
        user, game = users[u_idx], games[g_idx]
        rating, reason = -1, ''
        if game.platform not in ('PS', 'BOTH'):
            reason = 'Not on PlayStation'
        elif game.metacritic >= 70:
            if game.is_exclusive and game.platform == 'PS' and game.genre in ('Action', 'Adventure'):
                if random.random() < 0.65:
                    rating = 1; reason = 'PS exclusive Action/Adventure + good Metacritic (bonus)'
                else:
                    reason = 'PS exclusive but bonus did not trigger'
            else:
                rating = 1; reason = 'PS/BOTH + Metacritic >= 70'
        else:
            reason = 'PS platform but Metacritic too low'
        if random.random() < noise:
            rating *= -1; reason += ' [NOISE FLIP]'
        rows.append(_make_row(user, game, rating, reason))
    return rows


def generate_ratings_segment3(users, games, pairs, noise):
    """
    Generates ratings for Segment 3: Cross-Platform Gamer — Full Criteria.

    Criteria:
    - AVG_RATING >= 72 (Metacritic)
    - is_multiplayer = True
    - Game must be on BOTH platforms
    - BONUS: multiplayer + Metacritic 68-71 → 60% like chance
    - noise% chance of rating flip

    Args:
        users: full user list
        games: full game list
        pairs: (user_idx, game_idx) tuples for this segment
        noise: flip probability
    Returns:
        list of row dicts
    """
    rows = []
    for u_idx, g_idx in pairs:
        user, game = users[u_idx], games[g_idx]
        rating, reason = -1, ''
        if game.platform != 'BOTH':
            reason = 'Not on both platforms'
        elif game.metacritic >= 72 and game.is_multiplayer:
            rating = 1; reason = 'Cross-platform + Metacritic + multiplayer'
        elif game.is_multiplayer and 68 <= game.metacritic < 72:
            if random.random() < 0.60:
                rating = 1; reason = 'Multiplayer + decent Metacritic (bonus)'
            else:
                reason = 'Multiplayer but Metacritic borderline'
        else:
            reason = 'BOTH but Metacritic or multiplayer missing'
        if random.random() < noise:
            rating *= -1; reason += ' [NOISE FLIP]'
        rows.append(_make_row(user, game, rating, reason))
    return rows


def generate_ratings_segment4(users, games, pairs, noise):
    """
    Generates ratings for Segment 4: Budget Gamer — Full Criteria.

    Criteria:
    - AVG_RATING >= 55 (Metacritic)
    - price_eur <= 20
    - BONUS: price <= 10 + Metacritic >= 50 → 75% like chance
    - noise% chance of rating flip

    Args:
        users: full user list
        games: full game list
        pairs: (user_idx, game_idx) tuples for this segment
        noise: flip probability
    Returns:
        list of row dicts
    """
    rows = []
    for u_idx, g_idx in pairs:
        user, game = users[u_idx], games[g_idx]
        rating, reason = -1, ''
        if game.price_eur > 20:
            reason = 'Too expensive'
        elif game.metacritic >= 55:
            rating = 1; reason = 'Cheap + acceptable Metacritic'
        elif game.price_eur <= 10 and game.metacritic >= 50:
            if random.random() < 0.75:
                rating = 1; reason = 'Very cheap + borderline Metacritic (bonus)'
            else:
                reason = 'Very cheap but Metacritic still too low'
        else:
            reason = 'Cheap but Metacritic too low'
        if random.random() < noise:
            rating *= -1; reason += ' [NOISE FLIP]'
        rows.append(_make_row(user, game, rating, reason))
    return rows


def generate_ratings_segment5(users, games, pairs, noise):
    """
    Generates ratings for Segment 5: Casual / Family Gamer — Full Criteria.

    Criteria:
    - AVG_RATING >= 60 (Metacritic)
    - AVG_PLAYTIME <= 15h
    - age_rating in ('3', '7')
    - BONUS: Puzzle/Racing + family rating → 70% like chance
    - noise% chance of rating flip

    Args:
        users: full user list
        games: full game list
        pairs: (user_idx, game_idx) tuples for this segment
        noise: flip probability
    Returns:
        list of row dicts
    """
    rows = []
    for u_idx, g_idx in pairs:
        user, game = users[u_idx], games[g_idx]
        rating, reason = -1, ''
        if game.age_rating not in ('3', '7'):
            reason = 'Age rating too high for family'
        elif game.avg_playtime_h <= 15 and game.metacritic >= 60:
            rating = 1; reason = 'Family-friendly + short playtime + good Metacritic'
        elif game.genre in ('Puzzle', 'Racing') and game.age_rating in ('3', '7'):
            if random.random() < 0.70:
                rating = 1; reason = 'Casual genre + family rating (bonus)'
            else:
                reason = 'Casual genre but criteria not met'
        else:
            reason = 'Family rating but playtime or Metacritic too low'
        if random.random() < noise:
            rating *= -1; reason += ' [NOISE FLIP]'
        rows.append(_make_row(user, game, rating, reason))
    return rows

In [11]:
def generate_ratings(
    users: List[User],
    games: List[VideoGame],
    n_ratings: int = 10000,
    noise: float = 0.05,
    output_file: str = 'ratings_full.csv'
) -> None:
    """
    Generates binary like (+1) / dislike (-1) ratings using FULL criteria
    and writes them to a CSV file.

    Full like condition per segment:
        Segment 1 – PC Purist         : PC/BOTH + Metacritic>=75 + playtime>=20h
                                         Bonus: Strategy/RPG + Meta 70-74 → 70%
        Segment 2 – PlayStation Fan    : PS/BOTH + Metacritic>=70
                                         Bonus: PS exclusive Action/Adv. → 65%
        Segment 3 – Cross-Platform     : BOTH + Metacritic>=72 + is_multiplayer
                                         Bonus: multi + Meta 68-71 → 60%
        Segment 4 – Budget Gamer       : price<=20 + Metacritic>=55
                                         Bonus: price<=10 + Meta>=50 → 75%
        Segment 5 – Casual/Family      : age_rating in ('3','7') + playtime<=15 + Meta>=60
                                         Bonus: Puzzle/Racing → 70%

    After the ground-truth is decided, the rating is flipped with probability
    `noise`, simulating real-world inconsistent user behavior.

    Internally calls:
        generate_ratings_segment1() through generate_ratings_segment5()

    Args:
        users (List[User]):       User objects.
        games (List[VideoGame]):  VideoGame objects.
        n_ratings (int):          Total (user, game) pairs to rate. Default: 10000.
        noise (float):            Probability [0,1] of flipping a rating. Default: 0.05.
        output_file (str):        CSV filename. Default: 'ratings_full.csv'.

    Returns:
        None — writes results to output_file.
    """
    all_pairs = [(u, g) for u in range(len(users)) for g in range(len(games))]
    sampled   = random.sample(all_pairs, min(n_ratings, len(all_pairs)))

    seg_pairs = {s: [] for s in range(1, 6)}
    for u_idx, g_idx in sampled:
        seg_pairs[users[u_idx].segment].append((u_idx, g_idx))

    handlers = {
        1: generate_ratings_segment1,
        2: generate_ratings_segment2,
        3: generate_ratings_segment3,
        4: generate_ratings_segment4,
        5: generate_ratings_segment5,
    }

    all_rows = []
    for s in range(1, 6):
        all_rows.extend(handlers[s](users, games, seg_pairs[s], noise))
    random.shuffle(all_rows)

    fieldnames = ['segment','age','gender','game','platform','genre','price_eur',
                  'metacritic','avg_playtime_h','is_multiplayer','is_exclusive',
                  'age_rating','release_year','rating','reason']
    with open(output_file, 'w', newline='', encoding='utf-8') as fw:
        writer = csv.DictWriter(fw, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(all_rows)

    total    = len(all_rows)
    positive = sum(1 for r in all_rows if r['rating'] == 1)
    print(f'Total ratings    : {total:,}')
    print(f'Positive (like)  : {positive:,}  ({positive/total:.1%})')
    print(f'Negative (dislike): {total-positive:,}  ({(total-positive)/total:.1%})')
    print(f'Noise level      : {noise:.0%}')
    print(f'Saved to         : {output_file}')

In [12]:
generate_ratings(users, games, n_ratings=10000, noise=0.05, output_file='ratings_full.csv')

df = pd.read_csv('ratings_full.csv')
print()
print('Like rate per segment:')
for seg, grp in df.groupby('segment'):
    print(f'  Segment {seg} – {SEG_NAMES[seg]:25s}: {(grp["rating"]==1).mean():.1%} likes  ({len(grp):,} ratings)')

Total ratings    : 10,000
Positive (like)  : 2,051  (20.5%)
Negative (dislike): 7,949  (79.5%)
Noise level      : 5%
Saved to         : ratings_full.csv

Like rate per segment:
  Segment 1 – PC Purist                : 21.7% likes  (1,992 ratings)
  Segment 2 – PlayStation Fan          : 35.1% likes  (1,933 ratings)
  Segment 3 – Cross-Platform Gamer     : 11.5% likes  (2,077 ratings)
  Segment 4 – Budget Gamer             : 18.4% likes  (1,957 ratings)
  Segment 5 – Casual / Family Gamer    : 16.8% likes  (2,041 ratings)


In [13]:
df.head(10)

,segment,age,gender,game,platform,genre,price_eur,metacritic,avg_playtime_h,is_multiplayer,is_exclusive,age_rating,release_year,rating,reason
0,3,21,M,Game_0105,PC,Puzzle,23.06,61,67.0,False,True,7,2010,-1,Not on both platforms
1,2,23,F,Game_0020,BOTH,RPG,20.57,65,3.1,False,False,7,2018,-1,PS platform but Metacritic too low
2,4,20,F,Game_0030,PC,Strategy,37.32,87,40.7,False,True,18,2017,-1,Too expensive
3,2,37,M,Game_0028,PC,Simulation,40.89,72,19.0,True,True,3,2013,-1,Not on PlayStation
4,2,28,M,Game_0132,BOTH,RPG,54.13,79,7.8,True,False,3,2016,1,PS/BOTH + Metacritic >= 70
5,2,25,F,Game_0174,PC,Simulation,34.27,66,49.2,False,True,7,2018,1,Not on PlayStation [NOISE FLIP]
6,2,28,M,Game_0059,BOTH,Sports,39.97,77,23.3,False,False,3,2014,1,PS/BOTH + Metacritic >= 70
7,4,30,F,Game_0067,PC,Racing,68.31,58,8.6,False,True,16,2016,1,Too expensive [NOISE FLIP]
8,3,21,M,Game_0025,BOTH,RPG,25.46,91,49.8,False,False,12,2018,1,BOTH but Metacritic or multiplayer missing [NO...
9,3,23,F,Game_0058,PC,Horror,12.53,67,13.3,False,True,16,2022,-1,Not on both platforms


---
## 4. `learn_segments()` — LDA with Anchor Words

Same approach as Phase 1, but now the LIKE tokens are much more segment-specific
because only games that truly match the segment's personality get liked.

**Expected improvement over Phase 1:** The cross-tab diagonal should be stronger
— LDA should more cleanly separate all 5 segments, including Budget and Casual.

In [14]:
def learn_segments(
    ratings_file: str = 'ratings_full.csv',
    games: List[VideoGame] = None,
    k: int = 5,
    n_iter: int = 500,
    top_n_words: int = 10
) -> None:
    """
    Reads the ratings CSV and uses LDA (tomotopy) to discover the underlying
    customer segments from the like/dislike patterns.

    Method
    ------
    1. Each user → one DOCUMENT.
       Each rated game → TOKEN: "<title>_<platform>_<genre>_LIKE/DISLIKE"
    2. LDA is trained with k topics.
    3. ANCHOR WORDS seed each topic toward a known segment:
       - Topic 0: _PC_ tokens          → PC Purist
       - Topic 1: _PS_ tokens          → PlayStation Fan
       - Topic 2: _BOTH_ tokens        → Cross-Platform Gamer
       - Topic 3: cheap-game LIKE tokens → Budget Gamer
       - Topic 4: family-rated LIKE tokens → Casual/Family Gamer
    4. Top-N words per topic are printed with a summary.
    5. Cross-tab of true segment vs discovered topic is printed.

    Args:
        ratings_file (str):      Path to the CSV from generate_ratings().
        games (List[VideoGame]): Game objects (for anchor lookup).
        k (int):                 Number of LDA topics. Default: 5.
        n_iter (int):            Training iterations. Default: 500.
        top_n_words (int):       Top words per topic. Default: 10.

    Returns:
        None – prints topic summaries and cross-tab.
    """
    df = pd.read_csv(ratings_file)
    game_lookup = {g.title: g for g in games} if games else {}

    df['_uid'] = df['segment'].astype(str) + '_' + df['age'].astype(str) + '_' + df['gender']
    user_docs = {}
    for uid, grp in df.groupby('_uid'):
        tokens = [f"{r['game']}_{r['platform']}_{r['genre']}_{'LIKE' if r['rating']==1 else 'DISLIKE'}"
                  for _, r in grp.iterrows()]
        if tokens:
            user_docs[uid] = tokens

    print(f'Documents (unique users) : {len(user_docs)}')
    print(f'Avg tokens per document  : {np.mean([len(v) for v in user_docs.values()]):.1f}')

    all_tokens = set(t for doc in user_docs.values() for t in doc)
    anchor_pc   = [t for t in all_tokens if '_PC_' in t][:20]
    anchor_ps   = [t for t in all_tokens if '_PS_' in t][:20]
    anchor_both = [t for t in all_tokens if '_BOTH_' in t][:20]
    anchor_budget, anchor_casual = [], []
    if game_lookup:
        for t in all_tokens:
            parts = t.split('_')
            g_title = parts[0] + '_' + parts[1]
            if g_title not in game_lookup: continue
            g = game_lookup[g_title]
            if g.price_eur <= 20 and t.endswith('LIKE'):        anchor_budget.append(t)
            if g.age_rating in ('3','7') and t.endswith('LIKE'): anchor_casual.append(t)
    anchor_budget = anchor_budget[:20]
    anchor_casual = anchor_casual[:20]

    print(f'\nAnchor sizes: PC={len(anchor_pc)}, PS={len(anchor_ps)}, '
          f'BOTH={len(anchor_both)}, Budget={len(anchor_budget)}, Casual={len(anchor_casual)}')

    lda = tp.LDAModel(k=k, seed=42)
    for doc_tokens in user_docs.values():
        lda.add_doc(doc_tokens)
    print('\nTraining LDA...')
    for i in range(0, n_iter, 50):
        lda.train(50)
        print(f'  Iteration {i+50:4d}  |  log-likelihood: {lda.ll_per_word:.4f}')

    print('\n' + '='*65)
    print('  LDA DISCOVERED TOPICS')
    print('='*65)
    for tid in range(lda.k):
        top_words  = [p[0] for p in lda.get_topic_words(tid, top_n=top_n_words)]
        pc_c  = sum(1 for w in top_words if '_PC_' in w)
        ps_c  = sum(1 for w in top_words if '_PS_' in w)
        bot_c = sum(1 for w in top_words if '_BOTH_' in w)
        like_c = sum(1 for w in top_words if w.endswith('_LIKE'))
        dis_c  = sum(1 for w in top_words if w.endswith('_DISLIKE'))
        dominant = max({'PC':pc_c,'PS':ps_c,'BOTH':bot_c}, key=lambda x:{'PC':pc_c,'PS':ps_c,'BOTH':bot_c}[x])
        genres = [w.split('_')[3] for w in top_words if len(w.split('_'))>=5]
        top_genre = pd.Series(genres).value_counts().index[0] if genres else '?'
        print(f'\n--- Topic {tid} ---')
        print(f'  Dominant platform : {dominant}  (PC={pc_c}, PS={ps_c}, BOTH={bot_c})')
        print(f'  Top genre         : {top_genre}')
        print(f'  Sentiment         : {like_c} LIKE vs {dis_c} DISLIKE')
        print(f'  Top tokens:')
        for w in top_words: print(f'    {w}')

    print('\n' + '='*65)
    print('  True Segment vs Most-Probable LDA Topic')
    print('='*65)
    uid_list = list(user_docs.keys())
    rows = [{'true_segment': int(uid_list[i].split('_')[0]),
             'lda_topic': int(np.argmax(doc.get_topic_dist()))}
            for i, doc in enumerate(lda.docs)]
    ct = pd.crosstab(pd.DataFrame(rows)['true_segment'],
                     pd.DataFrame(rows)['lda_topic'],
                     rownames=['True Segment'], colnames=['LDA Topic'])
    print(ct)
    print('\nDone.')

In [15]:
learn_segments(ratings_file='ratings_full.csv', games=games, k=5, n_iter=500)

Documents (unique users) : 289
Avg tokens per document  : 34.6

Anchor sizes: PC=20, PS=20, BOTH=20, Budget=20, Casual=20

Training LDA...
  Iteration   50  |  log-likelihood: -6.0856
  Iteration  100  |  log-likelihood: -5.9595
  Iteration  150  |  log-likelihood: -5.9962
  Iteration  200  |  log-likelihood: -5.9818
  Iteration  250  |  log-likelihood: -5.9812
  Iteration  300  |  log-likelihood: -5.9741
  Iteration  350  |  log-likelihood: -5.9481
  Iteration  400  |  log-likelihood: -5.9482
  Iteration  450  |  log-likelihood: -5.9664
  Iteration  500  |  log-likelihood: -5.9468

  LDA DISCOVERED TOPICS

--- Topic 0 ---
  Dominant platform : PC  (PC=6, PS=4, BOTH=0)
  Top genre         : Action
  Sentiment         : 0 LIKE vs 10 DISLIKE
  Top tokens:
    Game_0184_PC_Puzzle_DISLIKE
    Game_0048_PS_Adventure_DISLIKE
    Game_0061_PS_Horror_DISLIKE
    Game_0116_PC_Action_DISLIKE
    Game_0125_PS_Adventure_DISLIKE
    Game_0021_PC_Action_DISLIKE
    Game_0037_PC_Puzzle_DISLIKE
    Ga

---
## Conclusion — Phase 2

With full criteria:
- The **like rate per segment drops** (stricter conditions = fewer likes, but more meaningful)
- Each segment's LIKE tokens are tightly coupled to specific attributes:
  - Segment 1: only high-Metacritic, long PC games
  - Segment 2: only well-reviewed PS/BOTH games
  - Segment 3: only BOTH + multiplayer games
  - Segment 4: only cheap + decent-Metacritic games
  - Segment 5: only short, family-rated, well-reviewed games
- The **LDA cross-tab shows a stronger diagonal** compared to Phase 1

**Key takeaway:** Harder criteria make each user's liked-game vocabulary more
segment-specific. Combined with anchor words that guide the topic-to-segment
alignment, LDA can successfully recover all 5 underlying customer segments.